# 22-13 · POST-Redirect-GET

Практика к разделу [«HTML-формы и отправка данных на сервер»](../../site/chapters/glava-22/22-13-formy.html).

## Цель

Убедиться, что обработчик формы отвечает редиректом (код 303 See Other), а не сразу готовой страницей.

## Рабочий пример

In [ ]:
from flask import Flask, redirect, request, url_for

app = Flask(__name__)
zapisi = []


@app.route("/")
def glavnaya():
    return f"Записей: {len(zapisi)}"


@app.route("/dobavit", methods=["POST"])
def dobavit():
    tekst = request.form.get("tekst", "").strip()
    if tekst:
        zapisi.append(tekst)
    # code=303 (See Other) точнее, чем редирект Flask по умолчанию (302
    # Found), описывает POST-Redirect-GET: результат смотрите по другому
    # адресу через GET.
    return redirect(url_for("glavnaya"), code=303)


client = app.test_client()
otvet = client.post("/dobavit", data={"tekst": "Первая запись"})

print("Код ответа:", otvet.status_code)
print("Заголовок Location:", otvet.headers.get("Location"))

## Проверка результата

In [ ]:
assert otvet.status_code == 303
assert otvet.headers.get("Location") is not None
assert zapisi == ["Первая запись"]
print("Верно: POST ответил редиректом 303, а не HTML-страницей напрямую.")

## Задание ★★ Самостоятельная задача

Пройдите по редиректу вручную: выполните GET на адрес из заголовка Location и убедитесь, что он показывает актуальное количество записей.

In [ ]:
otvet_posle_redirecta = client.get(otvet.headers["Location"])
telo = otvet_posle_redirecta.get_data(as_text=True)

assert "Записей: 1" in telo
print("Верно: страница после редиректа показывает уже обновлённые данные.")